In [1]:
## converting data
import pandas as pd
data = pd.read_csv('Housing.csv')
data.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [2]:
## convert variables
from sklearn.preprocessing import LabelEncoder
encoders = {}
binary_cols = [
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'prefarea'
]
label_encoder = LabelEncoder()
for col in binary_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

data


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,furnished
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,furnished
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,semi-furnished
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,furnished
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,1,0,1,0,0,2,0,unfurnished
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,semi-furnished
542,1750000,3620,2,1,1,1,0,0,0,0,0,0,unfurnished
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,furnished


In [3]:
## Encoding furnishing status

from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(handle_unknown="ignore",sparse_output=False)
furnshing_encoder = onehot_encoder.fit_transform(data[['furnishingstatus']])
furnshing_df = pd.DataFrame(furnshing_encoder, columns= onehot_encoder.get_feature_names_out(['furnishingstatus']))
furnshing_df

,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,1.0,0.0,0.0
1,1.0,0.0,0.0
2,0.0,1.0,0.0
3,1.0,0.0,0.0
4,1.0,0.0,0.0
...,...,...,...
540,0.0,0.0,1.0
541,0.0,1.0,0.0
542,0.0,0.0,1.0
543,1.0,0.0,0.0


In [4]:
data = pd.concat([data.drop('furnishingstatus',axis=1), furnshing_df], axis=1)
data

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,1.0,0.0,0.0
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,1.0,0.0,0.0
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,0.0,1.0,0.0
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,1.0,0.0,0.0
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,1,0,1,0,0,2,0,0.0,0.0,1.0
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,0.0,1.0,0.0
542,1750000,3620,2,1,1,1,0,0,0,0,0,0,0.0,0.0,1.0
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,1.0,0.0,0.0


In [5]:
data.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_furnished,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,1.0,0.0,0.0
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,1.0,0.0,0.0
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,0.0,1.0,0.0
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,1.0,0.0,0.0
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,1.0,0.0,0.0


In [6]:
import pickle
## Save the encoders and scaler
with open('label_encoder.pkl','wb') as file:
    pickle.dump(encoders,file)

with open('onehot_encoder_furnishingstatus.pkl','wb') as file:
    pickle.dump(onehot_encoder,file)

In [7]:
X = data.drop('price', axis=1)
Y = data['price']
X.head()
Y.head()

0    13300000
1    12250000
2    12250000
3    12215000
4    11410000
Name: price, dtype: int64

In [8]:
X
Y

0      13300000
1      12250000
2      12250000
3      12215000
4      11410000
         ...   
540     1820000
541     1767150
542     1750000
543     1750000
544     1750000
Name: price, Length: 545, dtype: int64

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2, random_state=42)

# Scale features
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)  # Use transform, NOT fit_transform!

# Scale target values (prices are in millions)
scaler_y = StandardScaler()
Y_train_scaled = scaler_y.fit_transform(Y_train.values.reshape(-1, 1)).flatten()
Y_test_scaled = scaler_y.transform(Y_test.values.reshape(-1, 1)).flatten()

with open('scalar.pkl','wb') as file:
    pickle.dump(scalar,file)

with open('scaler_y.pkl','wb') as file:
    pickle.dump(scaler_y,file)


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential

In [11]:
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [12]:
model = Sequential([
    Dense(64, activation = 'relu', input_shape = (X_train.shape[1],)),
    Dense(32, activation = 'relu'),
    Dense(1, activation = 'linear')  # Use linear for regression
])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                960       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3073 (12.00 KB)
Trainable params: 3073 (12.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


2025-12-30 22:30:46.975980: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2025-12-30 22:30:46.976007: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-12-30 22:30:46.976012: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-12-30 22:30:46.976033: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-30 22:30:46.976044: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [13]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()

model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

In [14]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

logs_dir =  "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir= logs_dir, histogram_freq=1)


In [15]:
## Setup early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [16]:
### Train the model (use scaled Y values)
history=model.fit(
    X_train, Y_train_scaled, validation_data=(X_test, Y_test_scaled), epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
 1/14 [=>............................] - ETA: 4s - loss: 0.7173 - mae: 0.7173

2025-12-30 22:30:47.377998: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


14/14 [==============================] - 1s 18ms/step - loss: 0.5287 - mae: 0.5287 - val_loss: 0.5853 - val_mae: 0.5853
Epoch 2/100
14/14 [==============================] - 0s 7ms/step - loss: 0.4325 - mae: 0.4325 - val_loss: 0.5589 - val_mae: 0.5589
Epoch 3/100
14/14 [==============================] - 0s 6ms/step - loss: 0.4167 - mae: 0.4167 - val_loss: 0.5757 - val_mae: 0.5757
Epoch 4/100
14/14 [==============================] - 0s 6ms/step - loss: 0.4113 - mae: 0.4113 - val_loss: 0.5627 - val_mae: 0.5627
Epoch 5/100
14/14 [==============================] - 0s 6ms/step - loss: 0.4123 - mae: 0.4123 - val_loss: 0.5537 - val_mae: 0.5537
Epoch 6/100
14/14 [==============================] - 0s 7ms/step - loss: 0.4182 - mae: 0.4182 - val_loss: 0.5718 - val_mae: 0.5718
Epoch 7/100
14/14 [==============================] - 0s 6ms/step - loss: 0.4145 - mae: 0.4145 - val_loss: 0.5603 - val_mae: 0.5603
Epoch 8/100
14/14 [==============================] - 0s 6ms/step - loss: 0.4138 - mae: 0.4138 

In [17]:
model.save("model.h5")

/Users/cds/Learning/ML/ann_home_price_prediction/venv/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
